# **Setup**

In [ ]:
# BRANCH = "step5-baseline"
# !rm -rf /tmp/oct-calibration && git clone -q -b {BRANCH} https://github.com/zyanislam/oct-calibration.git /tmp/oct-calibration
# import sys; sys.path.insert(0, "/tmp/oct-calibration")

# **Copy to local disk**

In [ ]:
# # Copy the 256x256 dataset to local disk once per session.
# # Kaggle's input disk waits ~30-60 ms the first time each file is read; local disk takes ~1 ms.
# import shutil
# import time
# from concurrent.futures import ThreadPoolExecutor
# from pathlib import Path

# import pandas as pd
# from tqdm.auto import tqdm

# SOURCE = Path("/kaggle/input/datasets/zyanislam/kermany-retinal-oct-v3-256")
# ROOT = Path("/tmp/oct256")   # local disk: fast, not saved with the notebook, wiped when the session ends

# paths = pd.read_csv("/tmp/oct-calibration/splits/oct_split_fold0.csv")["path"]

# def copy_one(rel):
#     dst = ROOT / rel
#     if not dst.exists():                       # rerunning in the same session skips copied files
#         dst.parent.mkdir(parents=True, exist_ok=True)
#         shutil.copyfile(SOURCE / rel, dst)

# t = time.time()
# with ThreadPoolExecutor(max_workers=64) as pool:   # many copies at once: each one mostly waits
#     list(tqdm(pool.map(copy_one, paths), total=len(paths), desc="Copying"))
# print(f"{len(paths)} files on local disk in {time.time() - t:.0f}s")

In [ ]:
# ROOT = SOURCE = Path("/kaggle/input/datasets/zyanislam/kermany-retinal-oct-v3-256")

# **Checks 2 and 3**

In [ ]:
# import torch
# from src.dataset import make_loaders
# from src.model import build_model
# from src.train import set_seed, first_loss, overfit_one_batch

# set_seed(42)
# device = torch.device("cuda")
# print("GPU:", torch.cuda.get_device_name(0))

# loaders = make_loaders(ROOT, batch_size=64, num_workers=4)
# model = build_model().to(device)

# print(f"Check 2 · first loss: {first_loss(model, loaders['train'], device):.3f}  (expect about 1.4–1.8)")
# losses = overfit_one_batch(model, loaders["train"], device, steps=50)
# print("Check 3 · overfit one batch:", [round(l, 3) for l in losses[::10]], "→ final", round(losses[-1], 4))

In [ ]:
# # Setup + checks 2 and 3, all in one cell: safe to run after any restart
# import importlib, subprocess, sys
# from pathlib import Path
# import torch

# # 1. Get the code
# BRANCH = "step5-baseline"
# REPO = Path("/tmp/oct-calibration")
# if not (REPO / "src" / "train.py").exists():
#     subprocess.run(["rm", "-rf", str(REPO)])
#     r = subprocess.run(["git", "clone", "-q", "-b", BRANCH,
#                         "https://github.com/zyanislam/oct-calibration.git", str(REPO)],
#                        capture_output=True, text=True)
#     assert r.returncode == 0, f"git clone failed. Is Internet on in Settings?\n{r.stderr}"
# print("code:", sorted(p.name for p in (REPO / "src").glob("*.py")))

# # Python caches "nothing here" for a path added before the folder existed: clear that
# sys.path_importer_cache.pop(str(REPO), None)
# importlib.invalidate_caches()
# if str(REPO) not in sys.path:
#     sys.path.insert(0, str(REPO))

# # 2. Point at the data (straight from the dataset: fine for one batch)
# ROOT = Path("/kaggle/input/datasets/zyanislam/kermany-retinal-oct-v3-256")
# assert ROOT.is_dir(), f"dataset not found at {ROOT}: attach it under Input"
# assert torch.cuda.is_available(), "no GPU: set Accelerator to GPU T4 x1 in Settings"

In [ ]:
# # 3. Checks
# from src.dataset import make_loaders
# from src.model import build_model
# from src.train import set_seed, first_loss, overfit_one_batch

# set_seed(42)
# device = torch.device("cuda")
# print("GPU:", torch.cuda.get_device_name(0))

# loaders = make_loaders(ROOT, batch_size=64, num_workers=4)
# model = build_model().to(device)

# print(f"Check 2 · first loss: {first_loss(model, loaders['train'], device):.3f}  (expect about 1.4–1.8)")
# losses = overfit_one_batch(model, loaders["train"], device, steps=50)
# print("Check 3 · overfit one batch:", [round(l, 3) for l in losses[::10]], "→ final", round(losses[-1], 4))

In [ ]:
# # Check 5 · smoke test: 3 epochs on 10,000 training scans. Self-contained.
# import importlib, shutil, subprocess, sys, time
# from concurrent.futures import ThreadPoolExecutor
# from pathlib import Path
# import pandas as pd, torch
# from tqdm.auto import tqdm

# # Code
# BRANCH, REPO = "step5-baseline", Path("/tmp/oct-calibration")
# if not (REPO / "src" / "train.py").exists():
#     subprocess.run(["rm", "-rf", str(REPO)])
#     r = subprocess.run(["git", "clone", "-q", "-b", BRANCH, "https://github.com/zyanislam/oct-calibration.git", str(REPO)],
#                        capture_output=True, text=True)
#     assert r.returncode == 0, f"git clone failed. Is Internet on?\n{r.stderr}"
# sys.path_importer_cache.pop(str(REPO), None); importlib.invalidate_caches()
# if str(REPO) not in sys.path: sys.path.insert(0, str(REPO))

# # Data on local disk (skips files already copied)
# SOURCE, ROOT = Path("/kaggle/input/datasets/zyanislam/kermany-retinal-oct-v3-256"), Path("/tmp/oct256")
# paths = pd.read_csv(REPO / "splits/oct_split_fold0.csv")["path"]
# def copy_one(rel):
#     dst = ROOT / rel
#     if not dst.exists():
#         dst.parent.mkdir(parents=True, exist_ok=True); shutil.copyfile(SOURCE / rel, dst)
# t = time.time()
# with ThreadPoolExecutor(max_workers=64) as pool:
#     list(tqdm(pool.map(copy_one, paths), total=len(paths), desc="Copying"))
# print(f"data ready in {time.time() - t:.0f}s")

# # Smoke test
# from src.dataset import make_loaders
# from src.model import build_model
# from src.train import set_seed, class_weights, subset_loader, fit

# assert torch.cuda.is_available(), "no GPU"
# set_seed(42)
# loaders = make_loaders(ROOT, batch_size=64, num_workers=4)
# weights = class_weights(loaders["train"].dataset.labels)
# print("class weights CNV/DME/DRUSEN/NORMAL:", [round(w, 2) for w in weights.tolist()])

# smoke = dict(loaders, train=subset_loader(loaders["train"], 10_000))
# log = fit(build_model(), smoke, "/kaggle/working/smoke", epochs=3, weights=weights)

In [ ]:
# Check 6 · full run: 8 epochs on all 97,479 training scans. Self-contained.
import importlib, shutil, subprocess, sys, time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
import pandas as pd, torch
from tqdm.auto import tqdm

# Code
BRANCH, REPO = "step5-baseline", Path("/tmp/oct-calibration")
if not (REPO / "src" / "train.py").exists():
    subprocess.run(["rm", "-rf", str(REPO)])
    r = subprocess.run(["git", "clone", "-q", "-b", BRANCH, "https://github.com/zyanislam/oct-calibration.git", str(REPO)],
                       capture_output=True, text=True)
    assert r.returncode == 0, f"git clone failed. Is Internet on?\n{r.stderr}"
sys.path_importer_cache.pop(str(REPO), None); importlib.invalidate_caches()
if str(REPO) not in sys.path: sys.path.insert(0, str(REPO))
print("code at", subprocess.run(["git", "-C", str(REPO), "log", "--oneline", "-1"], capture_output=True, text=True).stdout.strip())

# Data on local disk
SOURCE, ROOT = Path("/kaggle/input/datasets/zyanislam/kermany-retinal-oct-v3-256"), Path("/tmp/oct256")
paths = pd.read_csv(REPO / "splits/oct_split_fold0.csv")["path"]
def copy_one(rel):
    dst = ROOT / rel
    if not dst.exists():
        dst.parent.mkdir(parents=True, exist_ok=True); shutil.copyfile(SOURCE / rel, dst)
t = time.time()
with ThreadPoolExecutor(max_workers=64) as pool:
    list(tqdm(pool.map(copy_one, paths), total=len(paths), desc="Copying"))
print(f"data ready in {time.time() - t:.0f}s")

# Full run
from src.dataset import make_loaders
from src.model import build_model
from src.train import set_seed, class_weights, fit

assert torch.cuda.is_available(), "no GPU"
print("GPU:", torch.cuda.get_device_name(0), "| seed 42")
set_seed(42)
loaders = make_loaders(ROOT, batch_size=64, num_workers=4)
weights = class_weights(loaders["train"].dataset.labels)
log = fit(build_model(), loaders, "/kaggle/working/baseline", epochs=8, weights=weights)